# MicroPython Memory Profiling

This notebook demonstrates memory usage monitoring and profiling on MicroPython-enabled microcontrollers. It provides tools for tracking memory allocation patterns and identifying memory leaks during function execution.

## Key Components

### MemInfoLogger Utility Class
A decorator-based logging utility that captures memory state before and/or after function execution. Features include:

- **Decorator Mode**: Automatically logs memory info around function calls
- **Manual Logging**: Direct `MemInfoLogger.log()` calls for checkpoint measurements
- **Flexible Configuration**: 
    - `pre`: Log memory before function execution
    - `post`: Log memory after function execution (default: enabled)
    - `verbose`: Show detailed memory breakdown with `mem_info(1)`
- **Timestamp Tracking**: Combines nanosecond and millisecond precision timestamps

### Usage Patterns

```python
# As a decorator with default settings
@MemInfoLogger()
def my_function():
        pass

# With custom label
@MemInfoLogger(label="my_operation")
def complex_task():
        pass

# Manual logging at checkpoints
MemInfoLogger.log("after_initialization")
MemInfoLogger.log("detailed_check", verbose=True)
```

### Usage from other notebooks

It is possible to re-use the `MemInfoLogger` class from other notebooks without need to copy it.
You can use the 
`%run setup_meminfologger.ipynb` 
magic to run this notebook against the current selected MCU.



In [ ]:
# %%micropython
from micropython import mem_info
from time import ticks_ms, time_ns
import time
import random


class MemInfoLogger:
    """Decorator class for logging memory info before/after function execution.

    Can be used as:
        @MemInfoLogger()                        # label defaults to function name
        @MemInfoLogger(label="custom")          # custom label
        @MemInfoLogger(label="test", pre=True)  # with options

    Or call log directly:
        MemInfoLogger.log("checkpoint")
        MemInfoLogger.log("detailed", verbose=True)
    """

    def __init__(self, func=None, *, label=None, pre=False, post=True, verbose=False):
        # Detect incorrect usage: @MemInfoLogger without parentheses
        if func is not None:
            raise TypeError(
                "MemInfoLogger must be called with parentheses: "
                "@MemInfoLogger() not @MemInfoLogger"
            )
        self.label = label  # None means use function name
        self.pre = pre
        self.post = post
        self.verbose = verbose

    @staticmethod
    def timestamp():
        """Create a timestamp."""
        return time_ns() + ((ticks_ms() % 1_000_000) * 1_000)

    @staticmethod
    def log(label, verbose=False):
        """Core logging implementation."""
        print(f"\n*** Memory info {label} ***")
        print(f"time:{MemInfoLogger.timestamp()}")
        mem_info(1) if verbose else mem_info()
        print("***********************\n")

    def __call__(self, func):
        label = self.label if self.label is not None else func.__name__
        pre, post, verbose = self.pre, self.post, self.verbose

        def wrapper(*args, **kwargs):
            if pre:
                MemInfoLogger.log(f"{label} before", verbose)
            result = func(*args, **kwargs)
            if post:
                MemInfoLogger.log(f"{label} after", verbose)
            return result

        return wrapper


In [ ]:
print("MemInfoLogger set up.")